# 🎾 Padel Analytics — Système de Recommandation
## Section Avancée — Recommendation System

---
**Objectif :** Recommander à chaque joueur les tournois les plus adaptés à son profil de performance.

**Approches implémentées :**
- **Content-Based Filtering** — similarité cosinus entre profil joueur et profil tournoi
- **Collaborative Filtering** — joueurs similaires ont aimé les mêmes tournois (SVD)
- **Hybrid System** — combinaison pondérée des deux approches

**Évaluation :** Precision@K, Recall@K, NDCG@K, Coverage

---

## 📦 0. Installation

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn scipy -q
print('✅ Librairies installées')

## 📚 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from scipy.sparse import csr_matrix

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#7C3AED', '#0891B2', '#059669', '#D97706', '#DC2626']

print('✅ Imports réussis')

---
## 📖 2. Théorie — Qu'est-ce qu'un système de recommandation ?

Un système de recommandation répond à la question : **"Quel tournoi ce joueur devrait-il jouer ensuite ?"**

### 2.1 Content-Based Filtering
> On regarde le **profil du joueur** (taux de victoire, classement, style de jeu) et on cherche les **tournois qui correspondent** le mieux à ce profil.
> 
> Analogie : comme Netflix qui recommande des films similaires à ceux que tu as aimés.

### 2.2 Collaborative Filtering
> On regarde ce que **des joueurs similaires** ont bien performé, et on recommande les mêmes tournois.
> 
> Analogie : "Les joueurs qui te ressemblent ont très bien réussi dans ce tournoi."

### 2.3 Hybrid System
> Combinaison des deux pour avoir les avantages de chacun et compenser leurs faiblesses.

---
## 🧹 3. Chargement & Préparation des données

In [ ]:
df = pd.read_csv('fact_performanceF.csv')

print(f'Shape : {df.shape}')
print(f'Joueurs : {sorted(df["ID_player"].unique())}')
print(f'Tournois (id) : {df["id_tournament"].nunique()} uniques')
print(f'Années : {sorted(df["Year"].unique())}')
df.head(3)

In [ ]:
# ── Nettoyage ──────────────────────────────────────────────
df_clean = df.copy()

# Imputation des valeurs manquantes par médiane
cols_null = ['Nombre_de_spectateurs', 'scheduled_matches_count',
             'match_reservations_count', 'likes']
for col in cols_null:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Encodage Tier
df_clean['Tier_encoded'] = (df_clean['Tier'] == 'Top10').astype(int)

# ── Feature Engineering ────────────────────────────────────
df_clean['win_rate']        = df_clean['victoires'] / df_clean['matchs_joues'].replace(0, 1)
df_clean['points_per_match']= df_clean['points']    / df_clean['matchs_joues'].replace(0, 1)
df_clean['prize_money_avg'] = (df_clean['prize_money_min'] + df_clean['prize_money_max']) / 2
df_clean['cost_avg']        = (df_clean['real_cost_min']   + df_clean['real_cost_max'])   / 2
df_clean['roi']             = df_clean['prize_money_avg']  / df_clean['cost_avg'].replace(0, 1)
df_clean['fill_rate']       = df_clean['match_reservations_count'] / df_clean['scheduled_matches_count'].replace(0, 1)
df_clean['social_score']    = np.log1p(df_clean['likes']) + np.log1p(df_clean['Abonnes_Instagram_Novembre_2025'])

print(f'✅ Dataset nettoyé : {df_clean.shape}')
print(f'Nulls restants : {df_clean.isnull().sum().sum()}')

---
## 🏆 4. Construction des profils

### 4.1 Profil Joueur
Le profil d'un joueur = **ses performances moyennes** sur tous les tournois qu'il a joués.

In [ ]:
# Features pour construire les profils
PLAYER_FEATURES     = ['win_rate', 'points_per_match', 'classement_mondial',
                        'roi', 'social_score', 'Tier_encoded']
TOURNAMENT_FEATURES = ['prize_money_avg', 'Nombre_de_spectateurs', 'fill_rate',
                        'price', 'Tier_encoded', 'social_score']

# ── Profil joueur : moyenne de ses performances ─────────────
player_profiles = df_clean.groupby('ID_player')[PLAYER_FEATURES].mean()

print('Profil moyen des joueurs :')
print(player_profiles.round(3))

In [ ]:
# Visualisation des profils joueurs
fig, ax = plt.subplots(figsize=(12, 5))

# Normaliser pour visualisation radar-like
scaler_viz = MinMaxScaler()
profiles_norm = pd.DataFrame(
    scaler_viz.fit_transform(player_profiles),
    index=player_profiles.index,
    columns=player_profiles.columns
)

x = np.arange(len(PLAYER_FEATURES))
width = 0.2
colors = PALETTE[:len(player_profiles)]

for i, (pid, row) in enumerate(profiles_norm.iterrows()):
    ax.bar(x + i * width, row.values, width, label=f'Joueur {pid}',
           color=colors[i], alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(PLAYER_FEATURES, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Score normalisé (0-1)', fontsize=11)
ax.set_title('Profil normalisé des joueurs', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 4.2 Profil Tournoi
Le profil d'un tournoi = **ses caractéristiques moyennes** (prize money, spectateurs, niveau requis...).

In [ ]:
# Profil tournoi : moyenne des caractéristiques
tournament_profiles = df_clean.groupby('id_tournament')[TOURNAMENT_FEATURES].mean()

print(f'Nombre de tournois : {len(tournament_profiles)}')
print(tournament_profiles.head(5).round(3))

---
## 🔵 5. Modèle 1 — Content-Based Filtering (Similarité Cosinus)

**Principe :** Calculer la similarité entre le profil d'un joueur et le profil de chaque tournoi.
Le score de similarité cosinus mesure l'angle entre deux vecteurs :
- Score = 1 → parfaitement similaires
- Score = 0 → aucun lien
- Score = -1 → opposés

In [ ]:
# ── Normalisation des deux espaces de features ──────────────
# On doit aligner les features communes entre joueur et tournoi
COMMON_FEATURES = ['win_rate', 'prize_money_avg', 'Nombre_de_spectateurs',
                   'fill_rate', 'social_score', 'Tier_encoded']

# Reconstruire profil joueur avec les features communes
player_cb = df_clean.groupby('ID_player')[
    ['win_rate', 'prize_money_avg', 'Nombre_de_spectateurs',
     'fill_rate', 'social_score', 'Tier_encoded']
].mean()

tournament_cb = df_clean.groupby('id_tournament')[
    ['win_rate', 'prize_money_avg', 'Nombre_de_spectateurs',
     'fill_rate', 'social_score', 'Tier_encoded']
].mean()

# Normaliser
scaler_cb = StandardScaler()
all_data  = pd.concat([player_cb, tournament_cb], axis=0)
scaler_cb.fit(all_data)

player_cb_scaled     = scaler_cb.transform(player_cb)
tournament_cb_scaled = scaler_cb.transform(tournament_cb)

print(f'Matrice joueurs    : {player_cb_scaled.shape}')
print(f'Matrice tournois   : {tournament_cb_scaled.shape}')

In [ ]:
# ── Calcul de la similarité cosinus ─────────────────────────
# Shape : (n_joueurs, n_tournois)
similarity_matrix = cosine_similarity(player_cb_scaled, tournament_cb_scaled)

sim_df = pd.DataFrame(
    similarity_matrix,
    index=player_cb.index,
    columns=tournament_cb.index
)

print(f'Matrice de similarité : {sim_df.shape}')
print('Aperçu (joueurs × tournois) :')
print(sim_df.round(3))

In [ ]:
def recommend_content_based(player_id, top_k=5, already_played=True):
    """
    Recommande les top_k tournois pour un joueur donné.
    Si already_played=False, exclut les tournois déjà joués par ce joueur.
    """
    if player_id not in sim_df.index:
        print(f'Joueur {player_id} non trouvé.')
        return pd.DataFrame()

    scores = sim_df.loc[player_id].copy()

    if not already_played:
        # Exclure les tournois déjà joués
        played = df_clean[df_clean['ID_player'] == player_id]['id_tournament'].unique()
        scores = scores.drop(index=[t for t in played if t in scores.index], errors='ignore')

    top = scores.nlargest(top_k)

    result = pd.DataFrame({
        'id_tournament':    top.index,
        'similarity_score': top.values.round(4),
        'rank':             range(1, len(top) + 1)
    })

    # Enrichir avec les caractéristiques du tournoi
    result = result.merge(
        tournament_cb[['prize_money_avg', 'Nombre_de_spectateurs', 'Tier_encoded']],
        left_on='id_tournament', right_index=True, how='left'
    )
    return result


# Test pour tous les joueurs
print('=' * 60)
for pid in sorted(player_cb.index):
    reco = recommend_content_based(pid, top_k=3, already_played=False)
    print(f'\nTop 3 recommandations — Joueur {pid} :')
    if not reco.empty:
        print(reco[['rank', 'id_tournament', 'similarity_score',
                     'prize_money_avg', 'Nombre_de_spectateurs']].to_string(index=False))
print('=' * 60)

In [ ]:
# Heatmap de similarité joueurs × tournois
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    sim_df,
    annot=False, cmap='YlOrRd',
    xticklabels=[f'T{t}' for t in sim_df.columns],
    yticklabels=[f'J{p}' for p in sim_df.index],
    ax=ax, linewidths=0
)
ax.set_title('Similarité cosinus — Joueurs × Tournois (Content-Based)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Tournois')
ax.set_ylabel('Joueurs')
plt.tight_layout()
plt.show()

---
## 🟢 6. Modèle 2 — Collaborative Filtering (SVD)

**Principe :** Construire une matrice **joueur × tournoi** avec le score de performance (win_rate),
puis la décomposer avec SVD pour découvrir des patterns latents.

**SVD (Singular Value Decomposition)** = décompose la matrice en facteurs cachés.
Ex : un facteur latent pourrait représenter "tournois de haut niveau avec gros prize money".

In [ ]:
# ── Construction de la matrice joueur × tournoi ─────────────
# Valeur = win_rate du joueur dans ce tournoi
rating_matrix = df_clean.pivot_table(
    index='ID_player',
    columns='id_tournament',
    values='win_rate',
    aggfunc='mean'
)

print(f'Matrice joueur × tournoi : {rating_matrix.shape}')
print(f'Densité : {rating_matrix.notna().sum().sum() / rating_matrix.size * 100:.1f}%')
print(rating_matrix.round(3))

In [ ]:
# Remplir les NaN par 0 (tournois non joués = 0)
rating_filled = rating_matrix.fillna(0)

# ── SVD (Truncated) ─────────────────────────────────────────
N_COMPONENTS = min(3, min(rating_filled.shape) - 1)  # au max 3 facteurs latents

svd   = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
U     = svd.fit_transform(rating_filled.values)        # joueurs dans l'espace latent
Sigma = np.diag(svd.singular_values_)
Vt    = svd.components_                                # tournois dans l'espace latent

# Reconstruire la matrice complète (prédictions pour toutes les cases)
predicted_ratings = np.dot(U, Vt)
predicted_df = pd.DataFrame(
    predicted_ratings,
    index=rating_filled.index,
    columns=rating_filled.columns
)

print(f'SVD avec {N_COMPONENTS} composantes latentes')
print(f'Variance expliquée : {svd.explained_variance_ratio_.sum()*100:.1f}%')
print(f'\nMatrice prédite (win_rate estimé) :')
print(predicted_df.round(3))

In [ ]:
def recommend_collaborative(player_id, top_k=5, only_unplayed=True):
    """
    Recommande les tournois où le joueur devrait avoir le meilleur win_rate prédit.
    """
    if player_id not in predicted_df.index:
        print(f'Joueur {player_id} non trouvé.')
        return pd.DataFrame()

    scores = predicted_df.loc[player_id].copy()

    if only_unplayed:
        played = df_clean[df_clean['ID_player'] == player_id]['id_tournament'].unique()
        scores = scores.drop(index=[t for t in played if t in scores.index], errors='ignore')

    top = scores.nlargest(top_k)

    return pd.DataFrame({
        'id_tournament':    top.index,
        'predicted_winrate': top.values.round(4),
        'rank':              range(1, len(top) + 1)
    })


print('=' * 60)
for pid in sorted(rating_filled.index):
    reco = recommend_collaborative(pid, top_k=3)
    print(f'\nTop 3 (Collaborative) — Joueur {pid} :')
    if not reco.empty:
        print(reco.to_string(index=False))
print('=' * 60)

In [ ]:
# Visualisation : matrice réelle vs prédite
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

sns.heatmap(rating_matrix.fillna(0), annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'T{t}' for t in rating_matrix.columns],
            yticklabels=[f'J{p}' for p in rating_matrix.index],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Matrice réelle (win_rate)', fontsize=12, fontweight='bold')

sns.heatmap(predicted_df, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=[f'T{t}' for t in predicted_df.columns],
            yticklabels=[f'J{p}' for p in predicted_df.index],
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Matrice prédite SVD (win_rate estimé)', fontsize=12, fontweight='bold')

plt.suptitle('Collaborative Filtering — SVD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🟡 7. Modèle 3 — Système Hybride

**Principe :** Combiner les scores Content-Based et Collaborative avec des poids.

- alpha = 0.6 → 60% content-based + 40% collaborative
- On normalise chaque score entre 0 et 1 avant de combiner.

In [ ]:
def normalize_scores(scores_series):
    """Normalise une série de scores entre 0 et 1."""
    mn, mx = scores_series.min(), scores_series.max()
    if mx == mn:
        return scores_series * 0 + 0.5
    return (scores_series - mn) / (mx - mn)


def recommend_hybrid(player_id, top_k=5, alpha=0.6):
    """
    Recommandation hybride.
    alpha = poids du content-based (1-alpha = poids du collaborative)
    """
    # Scores content-based
    if player_id in sim_df.index:
        cb_scores = sim_df.loc[player_id].copy()
        cb_scores_norm = normalize_scores(cb_scores)
    else:
        cb_scores_norm = pd.Series(dtype=float)

    # Scores collaborative
    if player_id in predicted_df.index:
        cf_scores = predicted_df.loc[player_id].copy()
        cf_scores_norm = normalize_scores(cf_scores)
    else:
        cf_scores_norm = pd.Series(dtype=float)

    # Tournois en commun
    common_tournaments = cb_scores_norm.index.intersection(cf_scores_norm.index)

    hybrid_scores = (
        alpha * cb_scores_norm.loc[common_tournaments] +
        (1 - alpha) * cf_scores_norm.loc[common_tournaments]
    )

    # Exclure les tournois déjà joués
    played = df_clean[df_clean['ID_player'] == player_id]['id_tournament'].unique()
    hybrid_scores = hybrid_scores.drop(
        index=[t for t in played if t in hybrid_scores.index], errors='ignore'
    )

    top = hybrid_scores.nlargest(top_k)

    result = pd.DataFrame({
        'id_tournament': top.index,
        'hybrid_score':  top.values.round(4),
        'rank':          range(1, len(top) + 1)
    })

    # Ajouter cb et cf séparément pour comparaison
    result['cb_score'] = result['id_tournament'].map(
        cb_scores_norm.to_dict()).round(4)
    result['cf_score'] = result['id_tournament'].map(
        cf_scores_norm.to_dict()).round(4)

    return result


print('=' * 65)
print(f'Système Hybride (alpha=0.6 CB + 0.4 CF)')
print('=' * 65)
for pid in sorted(player_cb.index):
    reco = recommend_hybrid(pid, top_k=5, alpha=0.6)
    print(f'\nTop 5 recommandations — Joueur {pid} :')
    if not reco.empty:
        print(reco.to_string(index=False))
print('=' * 65)

In [ ]:
# Visualisation comparative des 3 approches pour un joueur
PLAYER_VIZ = sorted(player_cb.index)[0]  # Joueur 1

reco_cb  = recommend_content_based(PLAYER_VIZ, top_k=5, already_played=False)
reco_cf  = recommend_collaborative(PLAYER_VIZ, top_k=5)
reco_hyb = recommend_hybrid(PLAYER_VIZ, top_k=5)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Content-Based
if not reco_cb.empty:
    axes[0].barh([f'T{t}' for t in reco_cb['id_tournament']],
                 reco_cb['similarity_score'], color='#7C3AED', alpha=0.85, edgecolor='white')
    axes[0].set_title(f'Content-Based\nJoueur {PLAYER_VIZ}', fontweight='bold')
    axes[0].set_xlabel('Similarité cosinus')
    axes[0].invert_yaxis()

# Collaborative
if not reco_cf.empty:
    axes[1].barh([f'T{t}' for t in reco_cf['id_tournament']],
                 reco_cf['predicted_winrate'], color='#059669', alpha=0.85, edgecolor='white')
    axes[1].set_title(f'Collaborative Filtering\nJoueur {PLAYER_VIZ}', fontweight='bold')
    axes[1].set_xlabel('Win rate prédit')
    axes[1].invert_yaxis()

# Hybride
if not reco_hyb.empty:
    bars = axes[2].barh([f'T{t}' for t in reco_hyb['id_tournament']],
                        reco_hyb['hybrid_score'], color='#D97706', alpha=0.85, edgecolor='white')
    axes[2].set_title(f'Hybride (60% CB + 40% CF)\nJoueur {PLAYER_VIZ}', fontweight='bold')
    axes[2].set_xlabel('Score hybride')
    axes[2].invert_yaxis()
    for bar, val in zip(bars, reco_hyb['hybrid_score']):
        axes[2].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=9)

plt.suptitle(f'Comparaison des 3 approches — Joueur {PLAYER_VIZ}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📊 8. Évaluation des modèles

### Métriques utilisées :
- **Precision@K** : parmi les K recommandations, combien sont pertinentes ?
- **Recall@K** : parmi tous les tournois pertinents, combien sont dans les K recommandations ?
- **NDCG@K** : tient compte de l'ordre des recommandations (les meilleures en premier = mieux)
- **Coverage** : pourcentage de tournois distincts recommandés (diversité)

In [ ]:
def precision_at_k(recommended, relevant, k):
    """Proportion des K recommandations qui sont pertinentes."""
    rec_k = recommended[:k]
    hits  = len(set(rec_k) & set(relevant))
    return hits / k if k > 0 else 0


def recall_at_k(recommended, relevant, k):
    """Proportion des tournois pertinents retrouvés dans les K."""
    rec_k = recommended[:k]
    hits  = len(set(rec_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0


def ndcg_at_k(recommended, relevant, k):
    """Normalized Discounted Cumulative Gain — récompense les bons résultats en tête."""
    rec_k = recommended[:k]
    dcg   = sum(1 / np.log2(i + 2) for i, r in enumerate(rec_k) if r in relevant)
    idcg  = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0


def evaluate_system(recommend_func, players, df_clean, K=3, func_kwargs={}):
    """
    Évalue un système de recommandation en leave-one-out :
    - Pour chaque joueur, on cache son dernier tournoi joué
    - On vérifie si le système le recommande dans les K premiers
    """
    precisions, recalls, ndcgs = [], [], []
    all_recommended = set()

    for pid in players:
        player_data   = df_clean[df_clean['ID_player'] == pid].sort_values('id_Fact')
        if len(player_data) < 2:
            continue

        # Le dernier tournoi = item de test
        last_tournament = player_data.iloc[-1]['id_tournament']
        relevant        = [last_tournament]

        # Recommandations
        try:
            reco = recommend_func(pid, top_k=K, **func_kwargs)
            if reco.empty:
                continue
            # Récupérer la colonne de tournois
            reco_list = reco['id_tournament'].tolist()
        except Exception:
            continue

        precisions.append(precision_at_k(reco_list, relevant, K))
        recalls.append(recall_at_k(reco_list, relevant, K))
        ndcgs.append(ndcg_at_k(reco_list, relevant, K))
        all_recommended.update(reco_list)

    total_tournaments = df_clean['id_tournament'].nunique()
    coverage = len(all_recommended) / total_tournaments if total_tournaments > 0 else 0

    return {
        f'Precision@{K}': round(np.mean(precisions), 4) if precisions else 0,
        f'Recall@{K}':    round(np.mean(recalls), 4)    if recalls    else 0,
        f'NDCG@{K}':      round(np.mean(ndcgs), 4)      if ndcgs      else 0,
        'Coverage':        round(coverage, 4)
    }


players_list = sorted(df_clean['ID_player'].unique())
K = 3

metrics_cb  = evaluate_system(recommend_content_based, players_list, df_clean, K=K,
                               func_kwargs={'already_played': False})
metrics_cf  = evaluate_system(recommend_collaborative, players_list, df_clean, K=K,
                               func_kwargs={'only_unplayed': True})
metrics_hyb = evaluate_system(recommend_hybrid, players_list, df_clean, K=K,
                               func_kwargs={'alpha': 0.6})

df_eval = pd.DataFrame({
    'Content-Based':   metrics_cb,
    'Collaborative':   metrics_cf,
    'Hybride':         metrics_hyb
}).T

print('\n📊 ÉVALUATION DES MODÈLES DE RECOMMANDATION')
print('=' * 60)
print(df_eval.to_string())
print('=' * 60)

In [ ]:
# Visualisation des métriques
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

colors_models = ['#7C3AED', '#059669', '#D97706']
models        = df_eval.index.tolist()

for i, metric in enumerate(df_eval.columns):
    vals = df_eval[metric].values
    bars = axes[i].bar(models, vals, color=colors_models, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    axes[i].set_title(metric, fontsize=12, fontweight='bold')
    axes[i].set_ylim(0, max(vals) * 1.3 + 0.05)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Comparaison des 3 systèmes de recommandation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🎯 9. Analyse de l'impact du paramètre alpha (Hybride)

In [ ]:
# Tester différentes valeurs d'alpha
alphas  = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
results = []

for a in alphas:
    m = evaluate_system(recommend_hybrid, players_list, df_clean,
                        K=3, func_kwargs={'alpha': a})
    m['alpha'] = a
    results.append(m)

df_alpha = pd.DataFrame(results).set_index('alpha')

fig, ax = plt.subplots(figsize=(10, 5))
metric_col = f'NDCG@{K}'
ax.plot(df_alpha.index, df_alpha[metric_col], 'o-',
        color='#D97706', linewidth=2.5, markersize=8, label=metric_col)
ax.plot(df_alpha.index, df_alpha[f'Precision@{K}'], 's--',
        color='#7C3AED', linewidth=2, markersize=7, label=f'Precision@{K}')

best_alpha = df_alpha[metric_col].idxmax()
ax.axvline(x=best_alpha, color='red', linestyle=':', alpha=0.7,
           label=f'Meilleur alpha = {best_alpha}')
ax.set_xlabel('Alpha (poids Content-Based)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Impact du paramètre alpha sur les performances hybrides',
             fontsize=13, fontweight='bold')
ax.legend()
ax.set_xticks(alphas)
ax.set_xticklabels([f'{a}\n(CB={int(a*100)}%\nCF={int((1-a)*100)}%)' for a in alphas], fontsize=9)
plt.tight_layout()
plt.show()

print(f'\n✅ Meilleur alpha : {best_alpha} (NDCG@{K} = {df_alpha[metric_col].max():.4f})')

---
## 📋 10. Synthèse finale

In [ ]:
print('=' * 65)
print('🎾  PADEL ANALYTICS — SYSTÈME DE RECOMMANDATION')
print('=' * 65)
print()
print('3 modèles implémentés :')
print('  1. Content-Based  — similarité cosinus profil joueur/tournoi')
print('  2. Collaborative  — SVD sur matrice joueur × tournoi')
print('  3. Hybride        — combinaison pondérée des deux')
print()
print('Évaluation (Leave-One-Out, K=3) :')
print(df_eval.to_string())
print()
best_model = df_eval[f'NDCG@{K}'].idxmax()
print(f'✅ Meilleur modèle selon NDCG@{K} : {best_model}')
print(f'   Alpha optimal pour le hybride   : {best_alpha}')
print()
print('Exemple de recommandations finales (Hybride, meilleur alpha) :')
for pid in players_list:
    reco = recommend_hybrid(pid, top_k=3, alpha=best_alpha)
    if not reco.empty:
        tournois = reco['id_tournament'].tolist()
        scores   = reco['hybrid_score'].tolist()
        print(f'  Joueur {pid} → Tournois {tournois} (scores: {[round(s,3) for s in scores]})')
print('=' * 65)